<a href="https://colab.research.google.com/github/Lornatt88/maai901-vision-project/blob/main/notebooks/02_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Loading everything**

In [1]:
import os
import pandas as pd
import torch
import kagglehub
from google.colab import drive

path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/maai901_skin"
train_df = pd.read_csv(os.path.join(save_dir, "train.csv"))
val_df = pd.read_csv(os.path.join(save_dir, "val.csv"))
test_df = pd.read_csv(os.path.join(save_dir, "test.csv"))

print("Train:", len(train_df), "| Val:", len(val_df), "| Test:", len(test_df))
print("First image found:", os.path.exists(train_df["path"].iloc[0]))
print("GPU on:", torch.cuda.is_available())

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Mounted at /content/drive
Train: 2566 | Val: 542 | Test: 544
First image found: True
GPU on: True


### **Building the data pipeline**

**How:**
- Resize to 224 x 224 pixels: the size ViT and ResNet expect.

- Augmentation (training only): random flips and small rotations. A mole flipped upside down is still a mole, so this teaches the model to focus on the lesion, not its position. It also gives the model more variety without new photos.

- Normalise: adjusts the colour values to the scale the model was originally trained on.

In [2]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

classes = ["nv", "bkl", "mel", "bcc"]
class_to_idx = {c: i for i, c in enumerate(classes)}

class SkinDataset(Dataset):
    def __init__(self, df, transform):
        self.paths = df["path"].tolist()
        self.labels = [class_to_idx[d] for d in df["dx"]]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), self.labels[i]

def make_loaders(mean, std, batch_size=32):
    train_tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(20),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    train_loader = DataLoader(SkinDataset(train_df, train_tf), batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(SkinDataset(val_df, eval_tf), batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(SkinDataset(test_df, eval_tf), batch_size=batch_size, shuffle=False, num_workers=2)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = make_loaders([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
images, labels = next(iter(train_loader))
print("Batch shape:", images.shape)
print("First 10 labels:", labels[:10])

Batch shape: torch.Size([32, 3, 224, 224])
First 10 labels: tensor([2, 1, 1, 0, 0, 2, 1, 0, 1, 2])


### **The training code**

**balanced accuracy:** for each class, the % of real cases the model caught (that's called recall, or sensitivity in medicine). Then we average the 4.

In [3]:
import torch.nn as nn
import numpy as np
from sklearn.metrics import recall_score
import time, copy

device = "cuda"

counts = train_df["dx"].value_counts()
weights = torch.tensor([len(train_df) / (4 * counts[c]) for c in classes], dtype=torch.float).to(device)
print("Class weights:", dict(zip(classes, weights.cpu().numpy().round(2))))

def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, preds, trues = 0, [], []
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            if hasattr(out, "logits"):
                out = out.logits
            loss = criterion(out, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            preds += out.argmax(1).cpu().tolist()
            trues += y.cpu().tolist()
    acc = np.mean(np.array(preds) == np.array(trues))
    bal = recall_score(trues, preds, average="macro")
    return total_loss / len(loader.dataset), acc, bal

def train_model(model, train_loader, val_loader, lr, epochs, name):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    best_bal, best_state, history = 0, None, []
    for ep in range(1, epochs + 1):
        start = time.time()
        tr = run_epoch(model, train_loader, criterion, optimizer)
        va = run_epoch(model, val_loader, criterion)
        history.append({"epoch": ep, "train_loss": tr[0], "train_acc": tr[1], "train_bal_acc": tr[2],
                        "val_loss": va[0], "val_acc": va[1], "val_bal_acc": va[2]})
        print(f"Epoch {ep}/{epochs} | train loss {tr[0]:.3f} bal-acc {tr[2]:.3f} | "
              f"val loss {va[0]:.3f} bal-acc {va[2]:.3f} | {time.time() - start:.0f}s")
        if va[2] > best_bal:
            best_bal = va[2]
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    torch.save(best_state, os.path.join(save_dir, f"{name}.pt"))
    pd.DataFrame(history).to_csv(os.path.join(save_dir, f"{name}_history.csv"), index=False)
    print(f"Best validation balanced accuracy: {best_bal:.3f} | saved {name}.pt")
    return model

Class weights: {'nv': np.float32(0.98), 'bkl': np.float32(0.83), 'mel': np.float32(0.84), 'bcc': np.float32(1.71)}


### **Training Model 1, the frozen ViT**

**What we're doing:**

- Loading the ready-made ViT from Hugging Face (google/vit-base-patch16-224). It was trained on millions of everyday photos.
Replacing its last layer with a new one that outputs our 4 classes, instead of the 1,000 everyday categories it originally knew.

- Freezing everything except that new last layer.
"Freezing" = locking the model's settings so they can't change. Only the new layer learns.

In [ ]:
from transformers import ViTForImageClassification

vit_frozen = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=4,
    ignore_mismatched_sizes=True,
)

for name, param in vit_frozen.named_parameters():
    param.requires_grad = "classifier" in name

trainable = sum(p.numel() for p in vit_frozen.parameters() if p.requires_grad)
total = sum(p.numel() for p in vit_frozen.parameters())
print(f"Trainable settings: {trainable:,} out of {total:,}")

vit_frozen = train_model(vit_frozen, train_loader, val_loader, lr=1e-3, epochs=10, name="vit_frozen")

***Already ran this once and saved the results in (vit_frozen) on Google Drive.***

### **Train Model 2, the fully fine-tuned ViT**
**What we're doing:**

Same ViT, but this time nothing is frozen. All 86 million settings can adjust to skin images.

Three changes from Model 1:

- Smaller learning rate (0.00002 instead of 0.001): the model already knows a lot. Big steps would wreck what it learned from millions of photos. Small steps let it adjust gently. This is the standard setting for fine-tuning ViTs.

- Smaller batches (16 instead of 32 images per tray): training the whole model uses much more GPU memory. Smaller trays stop Colab from running out of memory and crashing.

- Clearing Model 1 from memory first: it's already saved in my Drive, so I remove it from the GPU to make room.

In [5]:
if "vit_frozen" in globals():
    del vit_frozen
torch.cuda.empty_cache()

train_loader, val_loader, test_loader = make_loaders([0.5, 0.5, 0.5], [0.5, 0.5, 0.5], batch_size=16)

vit_finetuned = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=4,
    ignore_mismatched_sizes=True,
)

trainable = sum(p.numel() for p in vit_finetuned.parameters() if p.requires_grad)
print(f"Trainable settings: {trainable:,}")

vit_finetuned = train_model(vit_finetuned, train_loader, val_loader, lr=2e-5, epochs=10, name="vit_finetuned")

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1000`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([4, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([4])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Trainable settings: 85,801,732
Epoch 1/10 | train loss 0.791 bal-acc 0.680 | val loss 0.650 bal-acc 0.768 | 98s
Epoch 2/10 | train loss 0.447 bal-acc 0.836 | val loss 0.649 bal-acc 0.707 | 98s
Epoch 3/10 | train loss 0.319 bal-acc 0.887 | val loss 0.611 bal-acc 0.752 | 99s
Epoch 4/10 | train loss 0.221 bal-acc 0.924 | val loss 0.579 bal-acc 0.763 | 99s
Epoch 5/10 | train loss 0.162 bal-acc 0.947 | val loss 0.603 bal-acc 0.756 | 98s
Epoch 6/10 | train loss 0.115 bal-acc 0.966 | val loss 0.626 bal-acc 0.798 | 99s
Epoch 7/10 | train loss 0.088 bal-acc 0.973 | val loss 0.677 bal-acc 0.765 | 98s
Epoch 8/10 | train loss 0.051 bal-acc 0.989 | val loss 0.656 bal-acc 0.806 | 98s
Epoch 9/10 | train loss 0.049 bal-acc 0.983 | val loss 0.674 bal-acc 0.794 | 98s
Epoch 10/10 | train loss 0.047 bal-acc 0.986 | val loss 0.752 bal-acc 0.807 | 98s
Best validation balanced accuracy: 0.807 | saved vit_finetuned.pt
